In [123]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
import unicodedata
import time
import pandas as pd

In [124]:
def set_up_driver():
    chrome_options = Options()
    #chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [125]:
def get_medicine_infomation(driver):
    
    info = {
        "hoat_chat": "",
        "quy_cach_dong_goi": "",
        "thuong_hieu": "",
        "xuat_xu": "",
        "thuoc_can_ke_toa": "",
        "dang_bao_che": "",
        "ham_luong": "",
        "nha_san_xuat": "",
        "tieu_chuan": "",
    }

    wait = WebDriverWait(driver, 0.5)
    for _ in range(4):
        if driver.find_elements(By.CSS_SELECTOR, "div.table-thongtinsanpham-border table.tbl-thongtinsanpham tbody tr"):
            break
        driver.execute_script("window.scrollBy(0, 350);")
        time.sleep(0.25)
    try:
        wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR, "div.table-thongtinsanpham-border table.tbl-thongtinsanpham tbody tr"
        )))

        rows = driver.find_elements(
            By.CSS_SELECTOR,
            "div.table-thongtinsanpham-border table.tbl-thongtinsanpham tbody tr"
        )

        raw = {}
        for row in rows:
            cells = row.find_elements(By.CSS_SELECTOR, "th, td")
            if len(cells) >= 2:
                k = cells[0].text.strip().rstrip(":")
                v = cells[1].text.strip()
                if k:
                    raw[k] = v

        # map label -> field cố định
        info["hoat_chat"] = raw.get("Hoạt chất", "")
        info["quy_cach_dong_goi"] = raw.get("Quy cách đóng gói", "")
        info["thuong_hieu"] = raw.get("Thương hiệu", "")
        info["xuat_xu"] = raw.get("Xuất xứ", "")
        info["thuoc_can_ke_toa"] = raw.get("Thuốc cần kê toa", "")
        info["dang_bao_che"] = raw.get("Dạng bào chế", "")
        info["ham_luong"] = raw.get("Hàm lượng", "")
        info["nha_san_xuat"] = raw.get("Nhà sản xuất", "")
        info["tieu_chuan"] = raw.get("Tiêu chuẩn", "")

    except Exception as e:
        print(f"Lỗi khi lấy bảng thông tin sản phẩm: {type(e).__name__}: {e}")

    return info


In [126]:
def expand_button(driver):
    try:
        wait = WebDriverWait(driver, 1)
        selector = "#main > div > div > div > div > div > div > div.col-12.col-md-8.sanpham_info.row > div.tab-sanpham.pt-4 > span"

       
        xem_them_btn = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, selector))
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center', inline: 'nearest'});",
            xem_them_btn
        )
        time.sleep(0.75)
        driver.execute_script("window.scrollBy(0, -120);")  
        time.sleep(0.75)

        
        wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
        try:
            xem_them_btn.click()
        except Exception:
            driver.execute_script("arguments[0].click();", xem_them_btn)

        time.sleep(0.5)

    except (NoSuchElementException, TimeoutException):
        print("Không tìm thấy nút 'Xem thêm'.")
        return None

In [127]:

from urllib.parse import urljoin

def get_image_links(driver):
    wait = WebDriverWait(driver, 0.25)
    image_urls = []
    seen = set()

    try:
        
        wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR, "div.sanpham_img, div.sanpham_img.overflow-hidden"
        )))

        
        img_elements = driver.find_elements(By.CSS_SELECTOR, """
            div.sanpham_img img,
            div.owl-carousel img,
            div.owl-stage-outer img,
            div.owl-item img
        """)

        for img in img_elements:
            
            src = (
                img.get_attribute("src")
                or img.get_attribute("data-src")
                or img.get_attribute("data-lazy")
                or img.get_attribute("data-original")
            )
            if not src:
                continue

            full_url = urljoin(driver.current_url, src.strip())
            
            if full_url.startswith("data:"):
                continue
            if any(x in full_url.lower() for x in ["logo", "icon", "zalo"]):
                continue

            if full_url not in seen:
                seen.add(full_url)
                image_urls.append(full_url)

       
        bg_elements = driver.find_elements(By.CSS_SELECTOR, "div.sanpham_img [style*='background-image']")
        for el in bg_elements:
            style = el.get_attribute("style") or ""
            if "url(" in style:
                raw = style.split("url(", 1)[1].split(")", 1)[0].strip("'\" ")
                if raw:
                    full_url = urljoin(driver.current_url, raw)
                    if full_url not in seen:
                        seen.add(full_url)
                        image_urls.append(full_url)

    except Exception as e:
        print(f"Lỗi khi lấy ảnh: {type(e).__name__}: {e}")

    return image_urls


In [128]:
def get_medicine_detail_info(driver):
    result = {
        "thanh_phan": "",
        "cong_dung_chi_dinh": "",
        "lieu_dung": "",
        "cach_dung": "",
        "luu_y": "",
        "chong_chi_dinh": "",
        "tac_dung_phu": "",
        "tuong_tac_thuoc": "",
        "Bao_quan": "",
    }

    wait = WebDriverWait(driver, 1)
    expand_button(driver)

    def norm_text(s: str) -> str:
        return unicodedata.normalize("NFC", (s or "")).strip().lower()

    def clean_text(s: str) -> str:
        return unicodedata.normalize("NFC", (s or "")).strip()

    def is_target_heading(text: str) -> bool:
        t = norm_text(text)
        return ("công dụng" in t) or ("chỉ định" in t)

    def is_stop_heading(text: str) -> bool:
        t = norm_text(text)
        return any(k in t for k in [
            "chống chỉ định", "liều dùng", "cách dùng", "tác dụng phụ",
            "thận trọng", "lưu ý", "tương tác", "bảo quản", "thành phần"
        ])

    try:
        root = wait.until(EC.visibility_of_element_located((
            By.CSS_SELECTOR, "#offcanvasbottom.show #data_detail_productid, #data_detail_productid"
        )))

        # Kéo thanh cuộn trong offcanvas xuống cuối để tải/hiện hết nội dung
        try:
            scroll_box = driver.find_element(
                By.CSS_SELECTOR,
                "#offcanvasbottom.show #load-canvas-bottom, #offcanvasbottom.show .offcanvas-body"
            )
        except NoSuchElementException:
            scroll_box = root

        last_top = -1
        for _ in range(20):
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollTop + 500;", scroll_box)
            time.sleep(0.2)
            cur_top = driver.execute_script("return arguments[0].scrollTop", scroll_box)
            max_top = driver.execute_script("return arguments[0].scrollHeight - arguments[0].clientHeight", scroll_box)
            if cur_top == last_top or cur_top >= max_top:
                break
            last_top = cur_top

        driver.execute_script("arguments[0].scrollTop = 0;", scroll_box)
        time.sleep(0.2)

        result["detail_raw_text"] = root.text.strip()

        # Duyệt theo thứ tự DOM để giữ đúng ngữ cảnh h2 -> h3 -> ul/li/p
        nodes = root.find_elements(By.XPATH, ".//*[self::h2 or self::h3 or self::p or self::li or self::ul]")
        current_h2 = None
        capture_section = False
        section_lines = []

        def flush_section():
            nonlocal section_lines
            content = "\n".join([x for x in section_lines if x]).strip()
            section_lines = []
            return content

        for el in nodes:
            tag = (el.tag_name or "").lower()
            txt = clean_text(el.get_attribute("textContent") or "")
            if not txt:
                continue

            if tag == "h2":
                # Nếu đang ở section mục tiêu mà gặp h2 khác thì kết thúc section
                if capture_section and not is_target_heading(txt):
                    break

                current_h2 = txt
                capture_section = is_target_heading(txt)
                if capture_section:
                    section_lines.append(txt)
                continue

            if not capture_section:
                continue

            # Đang ở trong section mục tiêu
            if tag == "h3":
                # Nếu gặp h3 là mục con, vẫn giữ trong section công dụng/chỉ định
                # nhưng nếu h3 thực chất là tiêu đề lớn khác thì dừng
                if is_stop_heading(txt):
                    break
                section_lines.append(txt)
                continue

            if tag in ("p", "li", "ul"):
                if is_stop_heading(txt):
                    break
                section_lines.append(txt)

        # Gán nội dung đã gom cho công dụng/chỉ định
        cong_dung_content = flush_section()
        if cong_dung_content:
            # Loại bỏ dòng tiêu đề đầu nếu có
            lines = cong_dung_content.split("\n")
            if lines and is_target_heading(lines[0]):
                lines = lines[1:]
            result["cong_dung_chi_dinh"] = "\n".join([x for x in lines if x]).strip()

        # Parse lại các section khác bằng cách quét tiêu đề chính
        nodes2 = root.find_elements(By.XPATH, ".//*[self::h2 or self::h3 or self::p or self::li]")
        current_title = None
        bucket = {}

        for el in nodes2:
            tag = (el.tag_name or "").lower()
            txt = clean_text(el.get_attribute("textContent") or "")
            if not txt:
                continue

            if tag == "h2":
                current_title = txt
                bucket.setdefault(current_title, [])
            else:
                if current_title is None:
                    current_title = "Mô tả"
                    bucket.setdefault(current_title, [])
                bucket[current_title].append(txt)

        for title, chunks in bucket.items():
            content = "\n".join(chunks).strip()
            t = norm_text(title)

            if "thành phần" in t:
                result["thanh_phan"] = content
            elif "liều dùng" in t:
                result["lieu_dung"] = content
            elif "cách dùng" in t:
                result["cach_dung"] = content
            elif "thận trọng" in t or "lưu ý" in t:
                result["luu_y"] = content
            elif "chống chỉ định" in t:
                result["chong_chi_dinh"] = content
            elif "tác dụng phụ" in t:
                result["tac_dung_phu"] = content
            elif "tương tác" in t and "thuốc" in t:
                result["tuong_tac_thuoc"] = content
            elif "bảo quản" in t:
                result["Bao_quan"] = content

        # Nếu parser ở trên chưa bắt được công dụng/chỉ định thì thử fallback theo bucket
        if not result["cong_dung_chi_dinh"]:
            for title, chunks in bucket.items():
                t = norm_text(title)
                if "công dụng" in t or "chỉ định" in t:
                    result["cong_dung_chi_dinh"] = "\n".join(chunks).strip()
                    break

    except Exception as e:
        print(f"Lỗi khi lấy nội dung chi tiết: {type(e).__name__}: {e}")

    return result

In [129]:
def get_medicine_data(driver):
    wait = WebDriverWait(driver, 1)

    try: 
        name = driver.find_element(By.XPATH, "//*[@id='main']/div/div/div/div/div/div/div[1]/div[2]/div[1]").text.strip()
        time.sleep(0.5)
    except Exception:
        name = ""

    try:
        root = wait.until(EC.visibility_of_element_located((
            By.CSS_SELECTOR, "#offcanvasbottom.show #data_detail_productid, #data_detail_productid"
        )))

        raw_text = root.text.strip()

        # Parse theo từng section h2
        sections = {}
        current_key = None

        children = root.find_elements(By.XPATH, "./*")
        for el in children:
            tag = (el.tag_name or "").lower()
            txt = el.text.strip()
            if not txt:
                continue

            if tag == "h2":
                current_key = txt
                if current_key not in sections:
                    sections[current_key] = ""
            else:
                if current_key is None:
                    current_key = "Mô tả chung"
                    if current_key not in sections:
                        sections[current_key] = ""
                sections[current_key] = (sections[current_key] + "\n" + txt).strip()

        return {
            "name": name,
            "raw_text": raw_text,
            "sections": sections
        }

    except Exception as e:
        print(f"Lỗi khi lấy nội dung chi tiết: {type(e).__name__}: {e}")
        return {"name": name, "raw_text": "", "sections": {}}

In [130]:
# detail = get_medicine_detail_info(driver)
# print(detail["sections"].get("Thành phần", ""))
# print(detail["sections"].get("Công dụng (Chỉ định)", ""))

In [131]:

def crawl_medicine_data(
    input_csv="medicine_links_deduplicated.csv",
    output_csv="minhchau_medicines_data.csv",
    temp_csv="minhchau_medicines_data_temp.csv",
    save_every=10,
    sleep_sec=1.5
):
    df = pd.read_csv(input_csv)

    driver = set_up_driver()
    results = []

    try:
        total = len(df)

        for i, row in df.iterrows():
            category = str(row.get("category", "")).strip()
            url = str(row.get("link", "")).strip()

            if not url:
                continue

            try:
                driver.get(url)
                WebDriverWait(driver, 1).until(
                    EC.presence_of_element_located((By.TAG_NAME, "body"))
                )
                time.sleep(sleep_sec)

                try:
                    name = driver.find_element(By.XPATH, "//*[@id='main']/div/div/div/div/div/div/div[1]/div[2]/div[1]").text.strip()
                except Exception:
                    name = ""
                time.sleep(0.5)

                # Bảng thông tin
                info = get_medicine_infomation(driver) or {}

                # Chi tiết xem thêm
                detail = get_medicine_detail_info(driver) or {}

                # Ảnh
                images = get_image_links(driver) or []

                # Kết hợp 2 cột gốc (category, link) + thông tin crawl mới
                record = {
                    "category": category,
                    "link": url,
                    "name": name,
                    "images": " | ".join(images),
                    "error": ""
                }

                record.update(info)
                if isinstance(detail, dict):
                    record.update(detail)
                else:
                    record["detail_raw_text"] = str(detail)

                results.append(record)
                print(f"[{i+1}/{total}] crawled: {name if name else url}")

            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                print(f"[{i+1}/{total}] error: {err}")
                results.append({
                    "category": category,
                    "link": url,
                    "name": "",
                    "images": "",
                    "error": err
                })

            if (i + 1) % save_every == 0:
                pd.DataFrame(results).to_csv(temp_csv, index=False, encoding="utf-8-sig")
                print(f"Saved temp: {len(results)} records -> {temp_csv}")

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    final_df = pd.DataFrame(results)
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\nĐã lưu {len(final_df)} sản phẩm vào {output_csv}")
    return final_df

In [132]:
# # Chạy hàm
# crawled_data = crawl_medicine_data()

In [133]:
prepro_df = pd.read_csv(r"D:\OU\KhoaLuanTotNghiep\DrugRecommandation\preprocessData_MinhChau\minhchau_medicines_data_cleaned.csv")
prepro_df.info()
prepro_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9237 entries, 0 to 9236
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   category            9237 non-null   object
 1   link                9237 non-null   object
 2   name                9211 non-null   object
 3   images              9212 non-null   object
 4   hoat_chat           4132 non-null   object
 5   quy_cach_dong_goi   8446 non-null   object
 6   thuong_hieu         7826 non-null   object
 7   xuat_xu             8713 non-null   object
 8   thuoc_can_ke_toa    4095 non-null   object
 9   dang_bao_che        4228 non-null   object
 10  nha_san_xuat        4237 non-null   object
 11  tieu_chuan          1366 non-null   object
 12  thanh_phan          8879 non-null   object
 13  cong_dung_chi_dinh  9004 non-null   object
 14  lieu_dung           6612 non-null   object
 15  cach_dung           5115 non-null   object
 16  luu_y               6456

,category,link,name,images,hoat_chat,quy_cach_dong_goi,thuong_hieu,xuat_xu,thuoc_can_ke_toa,dang_bao_che,...,thanh_phan,cong_dung_chi_dinh,lieu_dung,cach_dung,luu_y,chong_chi_dinh,tac_dung_phu,tuong_tac_thuoc,Bao_quan,detail_raw_text
0,Thuốc dùng ngoài,https://nhathuocminhchau.com.vn/hydq-cream-cos...,"Hydq Cream Cosmederma 20g - Kem trị nám, mờ thâm",https://cdn.famitaa.net/storage/uploads/noidun...,Hydroquinone,Hộp 1 tuýp 20g,Cosmederma Remedies,Ấn độ,Có,Kem,...,Hydroquinone 4%,"Điều trị những vùng da bị sạm như sạm, nám, tà...",Làm sạch vùng da cần điều trị rồi để da thật k...,NaN,Có thể chứa sulfites có thể gây phản ứng dị ứn...,Quá mẫn với thành phần của thuốc Sử dụng kem t...,"Thường gặp Kích ứng, đau hoặc sưng tại chỗ áp ...",Tránh sử dụng chung với Benzoyl Peroxide,"Nhiệt độ dưới 30 độ C, nơi khô ráo, tránh ánh ...",Thành phần Hydroquinone 4% Công dụng (Chỉ định...
1,Thuốc dùng ngoài,https://nhathuocminhchau.com.vn/trebor-cream-0...,"Trebor Cream 0,05 Hetero 20g - Gel trị mụn",https://cdn.famitaa.net/storage/uploads/noidun...,Tretinoin,Hộp 1 tuýp 20g,Hetero,Ấn độ,Có,Gel,...,Tretinoin 0.05%,"Được chỉ định trong điều trị mụn, lão hoá da d...",Thoa 20 phút sau khi da được làm sạch và khô h...,NaN,Thận trọng đối với bệnh chàm (có thể gây kích ...,Quá mẫn với Tretinoin hoặc bất cứ thành phần n...,Khô da quá mức Ban đỏ Căng da Ngứa Tăng sắc tố...,Tránh sử dụng chung với: Aminolevulinic acid d...,"Nhiệt độ dưới 30 độ C, nơi khô ráo, tránh ánh ...",Thành phần Tretinoin 0.05% Công dụng (Chỉ định...
2,"Thuốc tiêm, dịch truyền",https://nhathuocminhchau.com.vn/09-sodium-chlo...,0.9% Sodium Chloride Solution For I.V. Infutio...,https://cdn.famitaa.net/storage/uploads/noidun...,NaN,500ml,NaN,Philippine,NaN,NaN,...,Hoạt chất: Sodium Chloride 900mg Nước cất pha ...,Bổ sung Sodium chloride và nước trong các trườ...,Tiêm truyền tĩnh mạch Liều lượng do bác sĩ chỉ...,NaN,"Đặc biệt thận trọng ở bệnh nhân cao huyết áp, ...","Tình trạng ứ nước, tăng Sodium huyết. Hội chứn...",Các phản ứng phụ có thể xảy ra do dung dịch ho...,"Thừa Sodium làm tăng bài tiết Lithium, thiếu S...","Bảo quản nơi khô thoáng, tránh ánh nắng trực t...",Thành phần Hoạt chất: Sodium Chloride 900mg Nư...
3,"Thuốc tiêm, dịch truyền",https://nhathuocminhchau.com.vn/09-sodium-chlo...,0.9% Sodium Chloride Solution For I.V. Infutio...,https://cdn.famitaa.net/storage/uploads/noidun...,NaN,500ml,Euro-Med,Philippine,NaN,NaN,...,Hoạt chất: Sodium Chloride 900mg Nước cất pha ...,Bổ sung Sodium chloride và nước trong các trườ...,Tiêm truyền tĩnh mạch Liều lượng do bác sĩ chỉ...,NaN,"Đặc biệt thận trọng ở bệnh nhân cao huyết áp, ...","Tình trạng ứ nước, tăng Sodium huyết. Hội chứn...",Các phản ứng phụ có thể xảy ra do dung dịch ho...,"Thừa Sodium làm tăng bài tiết Lithium, thiếu S...","Bảo quản nơi khô thoáng, tránh ánh nắng trực t...",Công dụng của 0.9% Sodium Chloride Solution Fo...
4,Thuốc kháng Virus,https://nhathuocminhchau.com.vn/3d-het-heet-he...,3D Het Heet Health Care 30 viên,https://cdn.famitaa.net/storage/uploads/noidun...,Dolutegravir/Lamivudine/Tenofovir Disoproxil F...,30 viên,Heet Health Care,Ấn độ,Có,Viên nén,...,Tenofovir: 300mg Lamivudine: 300mg Dolutegravi...,Đây là viên kết hợp liều cố định của lamivudin...,Người lớn và thanh thiếu niên (từ 12 tuổi và c...,Dùng đường uống.,Dolutegravir + Lamivudine + Tenofovir không nê...,Chống chỉ định ở bệnh nhân quá mẫn với tenofov...,"Tenofovir disoproxil fumarate Thường gặp, ADR ...",Lamivudine: Nồng độ zidovudin trong huyết tươn...,"Nơi khô, tránh ánh sáng, nhiệt độ dưới 30°C.",Thành phần Tenofovir: 300mg Lamivudine: 300mg ...


In [134]:
# Lọc các link còn thiếu thông tin 'công dụng chỉ định'
missing_indication_df = prepro_df[
    (prepro_df["cong_dung_chi_dinh"].isna() | (prepro_df["cong_dung_chi_dinh"].str.strip() == ""))]
print(f"Số sản phẩm thiếu 'công dụng chỉ định': {len(missing_indication_df)}")
missing_indication_df[["link", "name", "cong_dung_chi_dinh"]].head(10)

Số sản phẩm thiếu 'công dụng chỉ định': 233


,link,name,cong_dung_chi_dinh
42,https://nhathuocminhchau.com.vn/recard-50-acar...,Acarbose 50mg Khánh Hòa,NaN
134,https://nhathuocminhchau.com.vn/afonat-100mg-s...,"Afonat 100mg Savipharma, Hộp 3 vỉ x 10 viên",NaN
135,https://nhathuocminhchau.com.vn/afonat-200mg-s...,"Afonat 200mg Savipharma, Hộp 3 vỉ x 10 viên",NaN
264,https://nhathuocminhchau.com.vn/akurit-4-sando...,"Akurit 4 Sandoz, 24 vỉ x 28 viên",NaN
369,https://nhathuocminhchau.com.vn/amiodarone-myl...,Amiodarone Mylan 200mg 30 viên,NaN
441,https://nhathuocminhchau.com.vn/thuoc-mylan-an...,"Anzavir R 300mg/100mg Mylan, Hộp 30 viên",NaN
479,https://nhathuocminhchau.com.vn/apixaban-table...,Apixaban Tablets 2.5mg Zydus Cadila 3 vỉ x 10 ...,NaN
485,https://nhathuocminhchau.com.vn/aplorar-hctz-3...,Aplorar HCTZ 300mg/12.5mg Abbott 3 vỉ x 10 viên,NaN
586,https://nhathuocminhchau.com.vn/atproton-20mg-...,"Atproton 20mg Macleods, Hộp 3 vỉ x 10 viên",NaN
620,https://nhathuocminhchau.com.vn/axe-brand-medi...,Axe Brand Medicated Oil 10ml dầu gió trắng cây...,NaN


In [135]:
# Lọc các link còn thiếu thông tin 'chống chỉ định'
missing_contra_df = prepro_df[
    (prepro_df["chong_chi_dinh"].isna() | (prepro_df["chong_chi_dinh"].str.strip() == ""))]
print(f"Số sản phẩm thiếu 'chống chỉ định': {len(missing_contra_df)}")
missing_contra_df[["link", "name", "chong_chi_dinh"]].head(10)

Số sản phẩm thiếu 'chống chỉ định': 968


,link,name,chong_chi_dinh
5,https://nhathuocminhchau.com.vn/a-borraginol-k...,A Borraginol Kokando 20 viên - Viên đặt trị trĩ,NaN
26,https://nhathuocminhchau.com.vn/survanta-suspe...,Abbvie SURVANTA SUSPENSION 25MG/ML,NaN
42,https://nhathuocminhchau.com.vn/recard-50-acar...,Acarbose 50mg Khánh Hòa,NaN
50,https://nhathuocminhchau.com.vn/thuoc-acemol-3...,Acemol 325mg Nadyphar 400 viên - Thuốc giảm đa...,NaN
59,https://nhathuocminhchau.com.vn/acetakan-120mg...,Acetakan 120 Agimexpharm 6 vỉ x 10 viên - Hỗ t...,NaN
61,https://nhathuocminhchau.com.vn/acetazolamid-a...,"Acetazolamid - Acetazolamid 250 mg, Hộp 10 vỉ ...",NaN
106,https://nhathuocminhchau.com.vn/adacast-60-lie...,Adacast 60 liều - Thuốc xịt mũi,NaN
109,https://nhathuocminhchau.com.vn/thuoc-adalat-60mg,"Adalat 60mg LA, Hộp 3 vỉ x 10 viên",NaN
116,https://nhathuocminhchau.com.vn/adapex-01-gel-...,"Adapex 0,1 Gel Hacks Slacks 30g - Gel trị mụn,...",NaN
134,https://nhathuocminhchau.com.vn/afonat-100mg-s...,"Afonat 100mg Savipharma, Hộp 3 vỉ x 10 viên",NaN


In [136]:
# Gộp các link thiếu 1 trong 2 'công dụng chỉ định' và 'chống chỉ định'
missing_info_df = prepro_df[
    (prepro_df["cong_dung_chi_dinh"].isna() | (prepro_df["cong_dung_chi_dinh"].str.strip() == "")) |
    (prepro_df["chong_chi_dinh"].isna() | (prepro_df["chong_chi_dinh"].str.strip() == ""))
]
print(f"Số sản phẩm thiếu 'công dụng chỉ định' hoặc 'chống chỉ định': {len(missing_info_df)}")
missing_info_df[["link", "name", "cong_dung_chi_dinh", "chong_chi_dinh"]].head(10)

Số sản phẩm thiếu 'công dụng chỉ định' hoặc 'chống chỉ định': 973


,link,name,cong_dung_chi_dinh,chong_chi_dinh
5,https://nhathuocminhchau.com.vn/a-borraginol-k...,A Borraginol Kokando 20 viên - Viên đặt trị trĩ,"Nhanh chóng làm giảm cơn đau nhức, ngứa rát, c...",NaN
26,https://nhathuocminhchau.com.vn/survanta-suspe...,Abbvie SURVANTA SUSPENSION 25MG/ML,Phòng ngừa & điều trị h/c suy hô hấp (bệnh màn...,NaN
42,https://nhathuocminhchau.com.vn/recard-50-acar...,Acarbose 50mg Khánh Hòa,NaN,NaN
50,https://nhathuocminhchau.com.vn/thuoc-acemol-3...,Acemol 325mg Nadyphar 400 viên - Thuốc giảm đa...,Hạ sốt và giảm đau trong các trường hợp: Cảm c...,NaN
59,https://nhathuocminhchau.com.vn/acetakan-120mg...,Acetakan 120 Agimexpharm 6 vỉ x 10 viên - Hỗ t...,Hỗ trợ trong các trường hợp thiểu năng tuần ho...,NaN
61,https://nhathuocminhchau.com.vn/acetazolamid-a...,"Acetazolamid - Acetazolamid 250 mg, Hộp 10 vỉ ...",Trị tăng nhãn áp. Phụ trị động kinh nhẹ.,NaN
106,https://nhathuocminhchau.com.vn/adacast-60-lie...,Adacast 60 liều - Thuốc xịt mũi,Điều trị triệu chứng viêm mũi theo mùa hoặc qu...,NaN
109,https://nhathuocminhchau.com.vn/thuoc-adalat-60mg,"Adalat 60mg LA, Hộp 3 vỉ x 10 viên",Nifedipine được sử dụng để điều trị cao huyết ...,NaN
116,https://nhathuocminhchau.com.vn/adapex-01-gel-...,"Adapex 0,1 Gel Hacks Slacks 30g - Gel trị mụn,...","Giảm mụn trứng cá. Giảm thâm. Giảm mụn viem ,s...",NaN
134,https://nhathuocminhchau.com.vn/afonat-100mg-s...,"Afonat 100mg Savipharma, Hộp 3 vỉ x 10 viên",NaN,NaN


In [137]:
import os
from datetime import datetime

LINK_COL = "link"
CHECKPOINT_CSV = "minhchau_missing_congdung_checkpoint.csv"
FIX_CSV = "minhchau_missing_congdung_recrawled.csv"
UPDATED_CSV = "minhchau_medicines_data_cleaned_updated.csv"

SAVE_EVERY = 10
SLEEP_SEC = 1.2
RESUME = True
MAX_ITEMS = None          # ví dụ: 30 để test nhanh
ONLY_FAILED = False       # True: chỉ chạy lại link failed/error trong checkpoint

def has_text(v):
    if v is None:
        return False
    s = str(v).strip().lower()
    return s not in ["", "none", "nan", "null", "n/a", "<na>"]

print("Đã nạp cấu hình recrawl.")

Đã nạp cấu hình recrawl.


In [138]:
work_df = missing_indication_df.copy()
work_df[LINK_COL] = work_df[LINK_COL].astype(str).str.strip()
work_df = work_df[work_df[LINK_COL] != ""].drop_duplicates(subset=[LINK_COL]).reset_index(drop=True)

if MAX_ITEMS is not None:
    work_df = work_df.head(MAX_ITEMS).copy()

processed_links = set()
checkpoint_rows = []
fix_rows = []

if RESUME and os.path.exists(CHECKPOINT_CSV):
    cp_df = pd.read_csv(CHECKPOINT_CSV)
    cp_df[LINK_COL] = cp_df[LINK_COL].astype(str).str.strip()
    checkpoint_rows = cp_df.to_dict("records")

    if ONLY_FAILED:
        processed_links = set(cp_df.loc[cp_df["status"].eq("success"), LINK_COL].tolist())
    else:
        processed_links = set(cp_df[LINK_COL].tolist())

print(f"Tổng link mục tiêu (thiếu cong_dung_chi_dinh): {len(work_df)}")
print(f"Đã có trong checkpoint: {len(processed_links)}")
print(f"Cần crawl tiếp: {len(work_df) - len(processed_links)}")

Tổng link mục tiêu (thiếu cong_dung_chi_dinh): 233
Đã có trong checkpoint: 0
Cần crawl tiếp: 233


In [139]:
# Cell 17 - Crawl từng link còn thiếu 'công dụng chỉ định' (vẫn thu thập cả chống chỉ định)
driver = None
try:
    driver = set_up_driver()
    total = len(work_df)

    for i, row in work_df.iterrows():
        link = str(row.get(LINK_COL, "")).strip()
        old_name = str(row.get("name", "")).strip()

        if (not link) or (link in processed_links):
            continue

        status = "failed"
        err = ""
        cong_dung_chi_dinh = ""
        chong_chi_dinh = ""
        detail_raw_text = ""
        crawled_name = ""

        try:
            driver.get(link)
            WebDriverWait(driver, 2).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(SLEEP_SEC)

            detail = get_medicine_detail_info(driver) or {}
            crawled_name = str(driver.title or "").strip()
            cong_dung_chi_dinh = str(detail.get("cong_dung_chi_dinh", "") or "").strip()
            chong_chi_dinh = str(detail.get("chong_chi_dinh", "") or "").strip()
            detail_raw_text = str(detail.get("detail_raw_text", "") or "").strip()

            # Chỉ cần có cong_dung_chi_dinh là tính success
            if has_text(cong_dung_chi_dinh):
                status = "success"
            elif has_text(chong_chi_dinh):
                # vẫn lưu lại để cập nhật chống chỉ định, nhưng không tính success cho mục tiêu chính
                status = "partial"

            fix_rows.append({
                LINK_COL: link,
                "name": old_name,
                "crawled_name": crawled_name,
                "cong_dung_chi_dinh": cong_dung_chi_dinh,
                "chong_chi_dinh": chong_chi_dinh,
                "detail_raw_text": detail_raw_text
            })
            print(f"[{i+1}/{total}] {status}: {link}")

        except Exception as e:
            status = "error"
            err = f"{type(e).__name__}: {e}"
            print(f"[{i+1}/{total}] error: {err} | {link}")

        checkpoint_rows.append({
            LINK_COL: link,
            'name': old_name,
            "status": status,
            "error": err,
            "cong_dung_chi_dinh": cong_dung_chi_dinh,
            "chong_chi_dinh": chong_chi_dinh
        })
        processed_links.add(link)

        if len(checkpoint_rows) % SAVE_EVERY == 0:
            pd.DataFrame(checkpoint_rows).drop_duplicates(subset=[LINK_COL], keep="last").to_csv(
                CHECKPOINT_CSV, index=False, encoding="utf-8-sig"
            )
            if len(fix_rows) > 0:
                pd.DataFrame(fix_rows).drop_duplicates(subset=[LINK_COL], keep="last").to_csv(
                    FIX_CSV, index=False, encoding="utf-8-sig"
                )
            print(f"Đã lưu tạm checkpoint: {len(checkpoint_rows)} dòng")
finally:
    if driver is not None:
        try:
            driver.quit()
        except Exception:
            pass
print("Xong bước crawl.")

Không tìm thấy nút 'Xem thêm'.
Lỗi khi lấy nội dung chi tiết: TimeoutException: Message: 

[1/233] failed: https://nhathuocminhchau.com.vn/recard-50-acarbose-50mg
Không tìm thấy nút 'Xem thêm'.
Lỗi khi lấy nội dung chi tiết: TimeoutException: Message: 

[2/233] failed: https://nhathuocminhchau.com.vn/afonat-100mg-savipharma-hop-3-vi-x-10-vien
Không tìm thấy nút 'Xem thêm'.
Lỗi khi lấy nội dung chi tiết: TimeoutException: Message: 

[3/233] failed: https://nhathuocminhchau.com.vn/afonat-200mg-savipharma-hop-3-vi-x-10-vien
Không tìm thấy nút 'Xem thêm'.
Lỗi khi lấy nội dung chi tiết: TimeoutException: Message: 

[4/233] failed: https://nhathuocminhchau.com.vn/akurit-4-sandoz-24-vi-x-28-vien
[5/233] success: https://nhathuocminhchau.com.vn/amiodarone-mylan-200mg-30-vien
Không tìm thấy nút 'Xem thêm'.
Lỗi khi lấy nội dung chi tiết: TimeoutException: Message: 

[6/233] failed: https://nhathuocminhchau.com.vn/thuoc-mylan-anzavir-r-300mg100mg-hop-30-vien
Không tìm thấy nút 'Xem thêm'.
Lỗi khi

In [140]:
checkpoint_df = pd.read_csv("minhchau_missing_congdung_checkpoint.csv")
checkpoint_df.head(20)

,link,name,status,error,cong_dung_chi_dinh,chong_chi_dinh
0,https://nhathuocminhchau.com.vn/recard-50-acar...,Acarbose 50mg Khánh Hòa,failed,NaN,NaN,NaN
1,https://nhathuocminhchau.com.vn/afonat-100mg-s...,"Afonat 100mg Savipharma, Hộp 3 vỉ x 10 viên",failed,NaN,NaN,NaN
2,https://nhathuocminhchau.com.vn/afonat-200mg-s...,"Afonat 200mg Savipharma, Hộp 3 vỉ x 10 viên",failed,NaN,NaN,NaN
3,https://nhathuocminhchau.com.vn/akurit-4-sando...,"Akurit 4 Sandoz, 24 vỉ x 28 viên",failed,NaN,NaN,NaN
4,https://nhathuocminhchau.com.vn/amiodarone-myl...,Amiodarone Mylan 200mg 30 viên,success,NaN,Dự phòng loạn nhịp thất và đột tử do ngừng tim...,Chống chỉ định trong Sốc tim.\nLoạn năng nút x...
5,https://nhathuocminhchau.com.vn/thuoc-mylan-an...,"Anzavir R 300mg/100mg Mylan, Hộp 30 viên",failed,NaN,NaN,NaN
6,https://nhathuocminhchau.com.vn/apixaban-table...,Apixaban Tablets 2.5mg Zydus Cadila 3 vỉ x 10 ...,failed,NaN,NaN,NaN
7,https://nhathuocminhchau.com.vn/aplorar-hctz-3...,Aplorar HCTZ 300mg/12.5mg Abbott 3 vỉ x 10 viên,failed,NaN,NaN,NaN
8,https://nhathuocminhchau.com.vn/atproton-20mg-...,"Atproton 20mg Macleods, Hộp 3 vỉ x 10 viên",failed,NaN,NaN,NaN
9,https://nhathuocminhchau.com.vn/axe-brand-medi...,Axe Brand Medicated Oil 10ml dầu gió trắng cây...,failed,NaN,NaN,NaN


In [141]:
# Cell 18 - Lưu file recrawl + checkpoint cuối
fix_df = pd.DataFrame(fix_rows) if len(fix_rows) > 0 else pd.DataFrame(columns=[
    LINK_COL, "name", "crawled_name", "cong_dung_chi_dinh", "chong_chi_dinh", "detail_raw_text"
])
fix_df = fix_df.drop_duplicates(subset=[LINK_COL], keep="last")

checkpoint_df = pd.DataFrame(checkpoint_rows) if len(checkpoint_rows) > 0 else pd.DataFrame(columns=[
    LINK_COL, "status", "error", "updated_at", "has_cong_dung_chi_dinh", "has_chong_chi_dinh"
])
checkpoint_df = checkpoint_df.drop_duplicates(subset=[LINK_COL], keep="last")

fix_df.to_csv(FIX_CSV, index=False, encoding="utf-8-sig")
checkpoint_df.to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")

print(f"Đã lưu: {FIX_CSV} ({len(fix_df)} dòng)")
print(f"Đã lưu: {CHECKPOINT_CSV} ({len(checkpoint_df)} dòng)")

Đã lưu: minhchau_missing_congdung_recrawled.csv (233 dòng)
Đã lưu: minhchau_missing_congdung_checkpoint.csv (233 dòng)


In [142]:
# Kiểm tra các link đã crawl được thông tin và các link vẫn còn thiếu thông tin
success_links = checkpoint_df.loc[checkpoint_df["status"].eq("success"), LINK_COL].tolist()
failed_links = checkpoint_df.loc[checkpoint_df["status"].ne("success"), LINK_COL].tolist()
print(f"Số link crawl thành công: {len(success_links)}")
print(f"Số link crawl thất bại: {len(failed_links)}")

Số link crawl thành công: 9
Số link crawl thất bại: 224


In [143]:
# Số link crawl thông công thông tin 'công dụng chỉ định'
success_cong_dung = checkpoint_df.loc[
    checkpoint_df["status"].eq("success") & checkpoint_df["cong_dung_chi_dinh"].apply(has_text),
    LINK_COL
].tolist()
print(f"Số link có 'công dụng chỉ định': {len(success_cong_dung)}")
success_cong_dung[:10]

Số link có 'công dụng chỉ định': 9


['https://nhathuocminhchau.com.vn/amiodarone-mylan-200mg-30-vien',
 'https://nhathuocminhchau.com.vn/kapeda-500mg-kocak-farma-12-vi-x-10-vien',
 'https://nhathuocminhchau.com.vn/mabthera-1400mg117ml-sc',
 'https://nhathuocminhchau.com.vn/thuoc-khang-sinh-koprodoxil-cefadroxil-500mg',
 'https://nhathuocminhchau.com.vn/thuoc-nadecin-10mg-hop-30-vien',
 'https://nhathuocminhchau.com.vn/pegaset-50-aurobindo-4-vi-x-7-vien',
 'https://nhathuocminhchau.com.vn/siro-tri-ho-ho-khan-toplexil-90ml',
 'https://nhathuocminhchau.com.vn/thuoc-dieu-tri-benh-ho-hap-hexinvon-8mg',
 'https://nhathuocminhchau.com.vn/vacomuc-100mg-vacopharm-30-goi']

In [144]:
# In thông tin chi tiết của một vài link thành công và thất bại để kiểm tra
print("\nMột vài link crawl thành công:")
fix_df.loc[fix_df[LINK_COL].isin(success_links), [LINK_COL, "name", "crawled_name", "cong_dung_chi_dinh", "chong_chi_dinh"]].head(5)



Một vài link crawl thành công:


,link,name,crawled_name,cong_dung_chi_dinh,chong_chi_dinh
4,https://nhathuocminhchau.com.vn/amiodarone-myl...,Amiodarone Mylan 200mg 30 viên,Amiodarone Mylan 200mg 30 viên - Chống loạn nh...,Dự phòng loạn nhịp thất và đột tử do ngừng tim...,Chống chỉ định trong Sốc tim.\nLoạn năng nút x...
53,https://nhathuocminhchau.com.vn/kapeda-500mg-k...,Kapeda 500mg Kocak Farma 12 vỉ x 10 viên,Kapeda 500mg Kocak Farma 12 vỉ x 10 viên,Ung thư vú\n\tUng thư đại trực tràng\n\tUng th...,"Quá mẫn cảm (dị ứng) với capecitabine, với bất..."
66,https://nhathuocminhchau.com.vn/mabthera-1400m...,MABTHERA 1400MG/11.7ML SC,MABTHERA 1400MG/11.7ML SC,U lympho không Hodgkin có độ ác thấp hoặc thể ...,"Tiền sử mẫn cảm với thành phần thuốc, với prot..."
127,https://nhathuocminhchau.com.vn/thuoc-khang-si...,Thuốc kháng sinh Koprodoxil Cefadroxil 500mg |...,Thuốc kháng sinh Koprodoxil Cefadroxil 500mg |...,Không dùng cho bệnh nhân mẫn cảm với kháng sin...,Không dùng cho bệnh nhân mẫn cảm với kháng sin...
147,https://nhathuocminhchau.com.vn/thuoc-nadecin-...,"Thuốc Nadecin 10mg, Hộp 30 viên","Thuốc Nadecin 10mg, Hộp 30 viên",Phòng và điều trị cơn đau thắt ngực.\n\tĐiều t...,"Huyết áp thấp, truỵ tim mạch.\nThiếu máu nặng...."


In [145]:
# Số link thành công nhưng thiếu thông tin chống chỉ định
success_but_missing_contra = fix_df[
    (fix_df[LINK_COL].isin(success_links)) &
    (~fix_df["chong_chi_dinh"].apply(has_text))
]
print(f"Số link crawl thành công nhưng vẫn thiếu 'chống chỉ định': {len(success_but_missing_contra)}")
success_but_missing_contra[[LINK_COL, "name", "crawled_name", "cong_dung_chi_dinh", "chong_chi_dinh"]].head(10)

Số link crawl thành công nhưng vẫn thiếu 'chống chỉ định': 0


,link,name,crawled_name,cong_dung_chi_dinh,chong_chi_dinh


In [146]:
print("\nMột vài link crawl thất bại:")
fix_df.loc[fix_df[LINK_COL].isin(failed_links), [LINK_COL, "name", "crawled_name", "cong_dung_chi_dinh", "chong_chi_dinh"]].head(5)


Một vài link crawl thất bại:


,link,name,crawled_name,cong_dung_chi_dinh,chong_chi_dinh
0,https://nhathuocminhchau.com.vn/recard-50-acar...,Acarbose 50mg Khánh Hòa,RECARD 50 – Acarbose 50mg,,
1,https://nhathuocminhchau.com.vn/afonat-100mg-s...,"Afonat 100mg Savipharma, Hộp 3 vỉ x 10 viên","Afonat 100mg Savipharma, Hộp 3 vỉ x 10 viên",,
2,https://nhathuocminhchau.com.vn/afonat-200mg-s...,"Afonat 200mg Savipharma, Hộp 3 vỉ x 10 viên","Afonat 200mg Savipharma, Hộp 3 vỉ x 10 viên",,
3,https://nhathuocminhchau.com.vn/akurit-4-sando...,"Akurit 4 Sandoz, 24 vỉ x 28 viên","Akurit 4 Sandoz, 24 vỉ x 28 viên",,
5,https://nhathuocminhchau.com.vn/thuoc-mylan-an...,"Anzavir R 300mg/100mg Mylan, Hộp 30 viên","Thuốc Mylan Anzavir R 300mg/100mg, Hộp 30 viên",,


In [151]:
# Cập nhật kết quả recrawl vào đúng bộ dữ liệu gốc đã làm sạch (prepro_df)
def normalize_link(u):
    u = str(u or "").strip()
    return u[:-1] if u.endswith("/") else u

base_df = prepro_df.copy()  # <-- dùng cleaned dataframe để đồng bộ với bước lọc thiếu ban đầu
base_df["link_norm"] = base_df[LINK_COL].apply(normalize_link)

fix_work = fix_df.copy()
fix_work["link_norm"] = fix_work[LINK_COL].apply(normalize_link)

# Chỉ giữ các dòng recrawl có dữ liệu thật sự
fix_work = fix_work[
    fix_work["cong_dung_chi_dinh"].apply(has_text) | fix_work["chong_chi_dinh"].apply(has_text)
].copy()

# Nếu có checkpoint thì ưu tiên chỉ dùng link status=success
if 'checkpoint_df' in globals() and len(checkpoint_df) > 0 and 'status' in checkpoint_df.columns:
    ok_links = set(
        checkpoint_df.loc[checkpoint_df["status"].eq("success"), LINK_COL]
        .astype(str).str.strip().apply(normalize_link).tolist()
    )
    fix_work = fix_work[fix_work["link_norm"].isin(ok_links)].copy()

# Trước khi cập nhật: thống kê thiếu
before_missing_cd = (base_df["cong_dung_chi_dinh"].isna() | (base_df["cong_dung_chi_dinh"].astype(str).str.strip() == "")).sum()
before_missing_ccd = (base_df["chong_chi_dinh"].isna() | (base_df["chong_chi_dinh"].astype(str).str.strip() == "")).sum()

fix_dict = fix_work.set_index("link_norm")[["cong_dung_chi_dinh", "chong_chi_dinh"]].to_dict("index")

updated_cd = 0
updated_ccd = 0

for idx, row in base_df.iterrows():
    link_norm = row.get("link_norm", "")
    if link_norm not in fix_dict:
        continue

    new_cd = fix_dict[link_norm].get("cong_dung_chi_dinh", "")
    new_ccd = fix_dict[link_norm].get("chong_chi_dinh", "")

    old_cd_empty = (pd.isna(row.get("cong_dung_chi_dinh")) or str(row.get("cong_dung_chi_dinh", "")).strip() == "")
    old_ccd_empty = (pd.isna(row.get("chong_chi_dinh")) or str(row.get("chong_chi_dinh", "")).strip() == "")

    if old_cd_empty and has_text(new_cd):
        base_df.at[idx, "cong_dung_chi_dinh"] = new_cd
        updated_cd += 1

    if old_ccd_empty and has_text(new_ccd):
        base_df.at[idx, "chong_chi_dinh"] = new_ccd
        updated_ccd += 1

# Sau cập nhật
after_missing_cd = (base_df["cong_dung_chi_dinh"].isna() | (base_df["cong_dung_chi_dinh"].astype(str).str.strip() == "")).sum()
after_missing_ccd = (base_df["chong_chi_dinh"].isna() | (base_df["chong_chi_dinh"].astype(str).str.strip() == "")).sum()

base_df.drop(columns=["link_norm"], inplace=True, errors="ignore")
base_df.to_csv(UPDATED_CSV, index=False, encoding="utf-8-sig")

print(f"Đã lưu file cập nhật: {UPDATED_CSV}")
print(f"Điền mới cong_dung_chi_dinh: {updated_cd}")
print(f"Điền mới chong_chi_dinh: {updated_ccd}")
print("\nThiếu trước -> sau:")
print(f"- cong_dung_chi_dinh: {before_missing_cd} -> {after_missing_cd}")
print(f"- chong_chi_dinh: {before_missing_ccd} -> {after_missing_ccd}")

Đã lưu file cập nhật: minhchau_medicines_data_cleaned_updated.csv
Điền mới cong_dung_chi_dinh: 9
Điền mới chong_chi_dinh: 5

Thiếu trước -> sau:
- cong_dung_chi_dinh: 233 -> 224
- chong_chi_dinh: 968 -> 963


In [152]:
base_df.info()
base_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9237 entries, 0 to 9236
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   category            9237 non-null   object
 1   link                9237 non-null   object
 2   name                9211 non-null   object
 3   images              9212 non-null   object
 4   hoat_chat           4132 non-null   object
 5   quy_cach_dong_goi   8446 non-null   object
 6   thuong_hieu         7826 non-null   object
 7   xuat_xu             8713 non-null   object
 8   thuoc_can_ke_toa    4095 non-null   object
 9   dang_bao_che        4228 non-null   object
 10  nha_san_xuat        4237 non-null   object
 11  tieu_chuan          1366 non-null   object
 12  thanh_phan          8879 non-null   object
 13  cong_dung_chi_dinh  9013 non-null   object
 14  lieu_dung           6612 non-null   object
 15  cach_dung           5115 non-null   object
 16  luu_y               6456

,category,link,name,images,hoat_chat,quy_cach_dong_goi,thuong_hieu,xuat_xu,thuoc_can_ke_toa,dang_bao_che,...,thanh_phan,cong_dung_chi_dinh,lieu_dung,cach_dung,luu_y,chong_chi_dinh,tac_dung_phu,tuong_tac_thuoc,Bao_quan,detail_raw_text
0,Thuốc dùng ngoài,https://nhathuocminhchau.com.vn/hydq-cream-cos...,"Hydq Cream Cosmederma 20g - Kem trị nám, mờ thâm",https://cdn.famitaa.net/storage/uploads/noidun...,Hydroquinone,Hộp 1 tuýp 20g,Cosmederma Remedies,Ấn độ,Có,Kem,...,Hydroquinone 4%,"Điều trị những vùng da bị sạm như sạm, nám, tà...",Làm sạch vùng da cần điều trị rồi để da thật k...,NaN,Có thể chứa sulfites có thể gây phản ứng dị ứn...,Quá mẫn với thành phần của thuốc Sử dụng kem t...,"Thường gặp Kích ứng, đau hoặc sưng tại chỗ áp ...",Tránh sử dụng chung với Benzoyl Peroxide,"Nhiệt độ dưới 30 độ C, nơi khô ráo, tránh ánh ...",Thành phần Hydroquinone 4% Công dụng (Chỉ định...
1,Thuốc dùng ngoài,https://nhathuocminhchau.com.vn/trebor-cream-0...,"Trebor Cream 0,05 Hetero 20g - Gel trị mụn",https://cdn.famitaa.net/storage/uploads/noidun...,Tretinoin,Hộp 1 tuýp 20g,Hetero,Ấn độ,Có,Gel,...,Tretinoin 0.05%,"Được chỉ định trong điều trị mụn, lão hoá da d...",Thoa 20 phút sau khi da được làm sạch và khô h...,NaN,Thận trọng đối với bệnh chàm (có thể gây kích ...,Quá mẫn với Tretinoin hoặc bất cứ thành phần n...,Khô da quá mức Ban đỏ Căng da Ngứa Tăng sắc tố...,Tránh sử dụng chung với: Aminolevulinic acid d...,"Nhiệt độ dưới 30 độ C, nơi khô ráo, tránh ánh ...",Thành phần Tretinoin 0.05% Công dụng (Chỉ định...
2,"Thuốc tiêm, dịch truyền",https://nhathuocminhchau.com.vn/09-sodium-chlo...,0.9% Sodium Chloride Solution For I.V. Infutio...,https://cdn.famitaa.net/storage/uploads/noidun...,NaN,500ml,NaN,Philippine,NaN,NaN,...,Hoạt chất: Sodium Chloride 900mg Nước cất pha ...,Bổ sung Sodium chloride và nước trong các trườ...,Tiêm truyền tĩnh mạch Liều lượng do bác sĩ chỉ...,NaN,"Đặc biệt thận trọng ở bệnh nhân cao huyết áp, ...","Tình trạng ứ nước, tăng Sodium huyết. Hội chứn...",Các phản ứng phụ có thể xảy ra do dung dịch ho...,"Thừa Sodium làm tăng bài tiết Lithium, thiếu S...","Bảo quản nơi khô thoáng, tránh ánh nắng trực t...",Thành phần Hoạt chất: Sodium Chloride 900mg Nư...
3,"Thuốc tiêm, dịch truyền",https://nhathuocminhchau.com.vn/09-sodium-chlo...,0.9% Sodium Chloride Solution For I.V. Infutio...,https://cdn.famitaa.net/storage/uploads/noidun...,NaN,500ml,Euro-Med,Philippine,NaN,NaN,...,Hoạt chất: Sodium Chloride 900mg Nước cất pha ...,Bổ sung Sodium chloride và nước trong các trườ...,Tiêm truyền tĩnh mạch Liều lượng do bác sĩ chỉ...,NaN,"Đặc biệt thận trọng ở bệnh nhân cao huyết áp, ...","Tình trạng ứ nước, tăng Sodium huyết. Hội chứn...",Các phản ứng phụ có thể xảy ra do dung dịch ho...,"Thừa Sodium làm tăng bài tiết Lithium, thiếu S...","Bảo quản nơi khô thoáng, tránh ánh nắng trực t...",Công dụng của 0.9% Sodium Chloride Solution Fo...
4,Thuốc kháng Virus,https://nhathuocminhchau.com.vn/3d-het-heet-he...,3D Het Heet Health Care 30 viên,https://cdn.famitaa.net/storage/uploads/noidun...,Dolutegravir/Lamivudine/Tenofovir Disoproxil F...,30 viên,Heet Health Care,Ấn độ,Có,Viên nén,...,Tenofovir: 300mg Lamivudine: 300mg Dolutegravi...,Đây là viên kết hợp liều cố định của lamivudin...,Người lớn và thanh thiếu niên (từ 12 tuổi và c...,Dùng đường uống.,Dolutegravir + Lamivudine + Tenofovir không nê...,Chống chỉ định ở bệnh nhân quá mẫn với tenofov...,"Tenofovir disoproxil fumarate Thường gặp, ADR ...",Lamivudine: Nồng độ zidovudin trong huyết tươn...,"Nơi khô, tránh ánh sáng, nhiệt độ dưới 30°C.",Thành phần Tenofovir: 300mg Lamivudine: 300mg ...


In [ ]:

def _norm_link(u):
    u = str(u or "").strip()
    return u[:-1] if u.endswith("/") else u

base_dbg = prepro_df.copy()
base_dbg["link_norm"] = base_dbg["link"].apply(_norm_link)

fix_dbg = fix_df.copy() if 'fix_df' in globals() else pd.read_csv(FIX_CSV)
fix_dbg["link_norm"] = fix_dbg["link"].apply(_norm_link)

cp_dbg = checkpoint_df.copy() if 'checkpoint_df' in globals() else pd.read_csv(CHECKPOINT_CSV)
cp_dbg["link_norm"] = cp_dbg["link"].astype(str).str.strip().apply(_norm_link)

# Chỉ xét link status success
success_norm = set(cp_dbg.loc[cp_dbg["status"].eq("success"), "link_norm"].tolist())
fix_success = fix_dbg[fix_dbg["link_norm"].isin(success_norm)].copy()

fix_success["has_cd"] = fix_success["cong_dung_chi_dinh"].apply(has_text)
fix_success["has_ccd"] = fix_success["chong_chi_dinh"].apply(has_text)

# Match được vào base không?
base_norm_set = set(base_dbg["link_norm"].tolist())
fix_success["matched_base"] = fix_success["link_norm"].isin(base_norm_set)

# Trường hợp có dữ liệu mới nhưng base đã có sẵn -> không tăng
base_lookup = base_dbg.set_index("link_norm")[["cong_dung_chi_dinh", "chong_chi_dinh"]]
def _old_empty(link_norm, col):
    if link_norm not in base_lookup.index:
        return None
    v = base_lookup.at[link_norm, col]
    return (pd.isna(v) or str(v).strip() == "")

fix_success["old_cd_empty"] = fix_success["link_norm"].apply(lambda x: _old_empty(x, "cong_dung_chi_dinh"))
fix_success["old_ccd_empty"] = fix_success["link_norm"].apply(lambda x: _old_empty(x, "chong_chi_dinh"))

can_fill_cd = fix_success["matched_base"] & fix_success["has_cd"] & fix_success["old_cd_empty"].fillna(False)
can_fill_ccd = fix_success["matched_base"] & fix_success["has_ccd"] & fix_success["old_ccd_empty"].fillna(False)

print("=== PHÂN TÍCH CHÊNH LỆCH ===")
print(f"Success links (checkpoint): {len(success_norm)}")
print(f"Success có record trong fix_df: {len(fix_success)}")
print(f"Success có thể match vào base: {fix_success['matched_base'].sum()}")
print(f"Success KHÔNG match vào base: {(~fix_success['matched_base']).sum()}")
print()
print(f"Success có cong_dung_chi_dinh: {fix_success['has_cd'].sum()}")
print(f"Success có chong_chi_dinh: {fix_success['has_ccd'].sum()}")
print()
print(f"Có thể điền MỚI cong_dung_chi_dinh: {can_fill_cd.sum()}")
print(f"Có thể điền MỚI chong_chi_dinh: {can_fill_ccd.sum()}")

print("\nTop link success nhưng không thể điền mới chống chỉ định (thường do vẫn rỗng):")
display(fix_success.loc[~can_fill_ccd, ["link", "cong_dung_chi_dinh", "chong_chi_dinh"]].head(10))

=== PHÂN TÍCH CHÊNH LỆCH ===
Success links (checkpoint): 9
Success có record trong fix_df: 9
Success có thể match vào base: 9
Success KHÔNG match vào base: 0

Success có cong_dung_chi_dinh: 9
Success có chong_chi_dinh: 9

Có thể điền MỚI cong_dung_chi_dinh: 9
Có thể điền MỚI chong_chi_dinh: 5

Top link success nhưng không thể điền mới chống chỉ định (thường do vẫn rỗng):


,link,cong_dung_chi_dinh,chong_chi_dinh
4,https://nhathuocminhchau.com.vn/amiodarone-myl...,Dự phòng loạn nhịp thất và đột tử do ngừng tim...,Chống chỉ định trong Sốc tim.\nLoạn năng nút x...
53,https://nhathuocminhchau.com.vn/kapeda-500mg-k...,Ung thư vú\n\tUng thư đại trực tràng\n\tUng th...,"Quá mẫn cảm (dị ứng) với capecitabine, với bất..."
66,https://nhathuocminhchau.com.vn/mabthera-1400m...,U lympho không Hodgkin có độ ác thấp hoặc thể ...,"Tiền sử mẫn cảm với thành phần thuốc, với prot..."
127,https://nhathuocminhchau.com.vn/thuoc-khang-si...,Không dùng cho bệnh nhân mẫn cảm với kháng sin...,Không dùng cho bệnh nhân mẫn cảm với kháng sin...
